# Can the context predict how EI's run ends, and does gating the UCB deviation by it beat the static schedule?

## Description

<!-- Written 2026-11-12 16:00 before the first run (pre-registration). Follows notebook 13 (option_policy, INCONCLUSIVE) and Tim's mandate of 2026-11-12 ("do both"). Issue #13. -->

**Question.** In notebook 13 the one-deviation policy beat EI only by as much as the static schedule "UCB(0.995) from t = 2" does. UCB helps a lot where EI ends badly and hurts where EI ends well. So:
1. **Probe.** Does the context at decision step t predict how EI's run will end (its terminal log10 regret)? Does the PFN's belief add to plain statistics of the data?
2. **Gate.** Does "deviate to UCB only if EI is predicted to end badly" beat the best static schedule?
3. **Ceiling.** What would such a gate gain if it *knew* EI's outcome on this task, luck-corrected? This bounds the idea whatever the predictor.

**What is already known** (scratch on notebook 13's pilot labels, run artifact `labels.pt` of `pilot_20261112-123724`; exploratory, the reason for this experiment, to be confirmed here on fresh tasks):
- EI's terminal regret is partly predictable: Spearman of a cross-fitted prediction with the outcome is 0.29–0.33 at t = 0, 0.45 at t = 2, 0.62 at t = 6, 0.71 at t = 14 with belief features; with data features only 0.22, 0.33, 0.60, 0.71. The current true regret (privileged) reaches 0.56 at t = 2 and 0.95 at t = 14.
- A gate on that prediction chose "always deviate" in every fold: gain over the static schedule 0.000.
- **Correction to notebook 13's post hoc.** There, giving the policy EI's own terminal regret as a feature gained +0.147 over static. That number is mostly mechanical: the same run's regret is half of the label. Gating by EI's outcome under the *other* candidate-set seed gives +0.156 [0.116, 0.201] at t = 2 against +0.131 for always deviating.
- By tercile of the other seed's EI outcome, "UCB for the rest" from t = 2 is worth −0.08, +0.02, +0.46: the hard third carries the whole gain, the loss on the easy third is small.

**Setup** (`tasks/ei_failure.py`, `analysis/ei_failure.py`; labels and context from `tasks/option_policy.py::collect`). d = 2, n_init = 4, T = 20, pfn_large, fp32, noise-free prior tasks, deterministic arms on seeded candidate sets (as notebooks 12–14).
- *Tasks.* 3000 fresh prior draws, split 4, task offset 50000.
- *Runs.* EI under seed 0 for the whole run. At every t ∈ {0, 2, …, 14}, from EI's data, under seeds 0 and 1: EI to T (the base), "UCB for 3 steps, then EI", "UCB for the rest".
- *Target* g: log10 terminal regret (ε = 1e-6) of the EI run under seed 0. *Luck replicate*: EI continued from step t under seed 1 (same data up to t, another candidate set afterwards).
- *Predictors* ĝ, cross-fitted over tasks (5 folds), from the feature sets `data` (18), `belief` (data + 45 belief features), `pfn` (belief + 32 PCA components of the PFN state): a ridge per decision step, and one MLP over all steps.
- *Gate.* Deviate at step j to option o iff score_j ≥ c. (j, o, c) are chosen on the training folds; for ĝ from predictions those tasks' models never saw. c = −∞ is the ungated static schedule and never deviating is allowed; both win ties.
  - deployable scores: ĝ of each feature set and learner. **Primary: `gate/belief/mlp`.**
  - privileged scores: `priv:other_seed` (EI's outcome from step j under seed 1; chosen and valued on seed-0 advantages: the luck-corrected ceiling); `priv:same_seed` (the target itself: mechanical, shown to make that visible); `priv:current_regret`.
- *Comparators.* EI (0); the best static (step, option) chosen on the training folds.

**What to look for.**
1. Probe: Spearman(ĝ, g) per decision step with bootstrap CIs; `belief` − `data` and `pfn` − `belief`, paired; against `priv:other_seed` (what any predictor could reach) and `priv:current_regret`.
2. Strata: the advantage of the UCB options by tercile of ĝ and of the other seed's outcome, per step.
3. Gates: gain over EI and over the static schedule, seed 0 (deployed) and seed 1 (executed on another candidate set).

**GO / NO-GO** (rule `analysis/option_policy.py::policy_verdict` on `gate/belief/mlp`, seed 0; worthwhile = 0.022 log10).
- **GO**: the gate beats EI (CI > 0) and beats the static schedule (CI > 0, ≥ 0.022). Then put ĝ into the option policy.
- **NO-GO**: the upper CI bound of the gate's gain over the static schedule is < 0.022. Expected from the exploration.
- **INCONCLUSIVE**: otherwise.
- Stated in advance for the ceiling: if `gate/priv:other_seed` gains less than 0.05 over the static schedule (upper CI bound), predicting EI's failure cannot make a single gated UCB deviation worthwhile, however good the predictor. The probe result stands on its own either way.

**Limits.**
- Noise-free P0, d = 2, T = 20. Only UCB options; one deviation.
- The luck replicate shares the data up to step t, so early steps are close to an independent run and late steps are not.
- The exploration above used notebook 13's tasks; the tasks here are fresh. Design choices (scores, grid, primary) were fixed before the first run.
- Labels carry GPU floating-point noise at the scale of ε (notebook 13).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anytime_acq.analysis import load_snapshot, show_header
from anytime_acq.analysis.option_policy import policy_verdict

SNAPSHOT = "results/ei_failure/<sweep_id>"  # set after `just snapshot`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

In [ ]:
runs = snap.runs
print("member status:", runs.status.value_counts().to_dict())
print("minutes:", {k: round(float(runs.iloc[0][k]) / 60, 1) for k in runs.columns if k.startswith("time/")})
summ, probe, pdiff, strata, gates, per, opts = (snap.table(k) for k in ("summary", "probe", "probe_diff", "strata", "gates", "per_task", "options"))
PRIMARY, WORTH = "gate/belief/mlp", 0.022
print("tasks:", len(per), "| EI terminal log10 regret: mean %.2f, sd %.2f" % (per.g0.mean(), per.g0.std()))
print("mean advantage over EI per step and option (seed 0 / seed 1):")
display(opts.pivot(index="option", columns="step", values="adv0").round(3)); display(opts.pivot(index="option", columns="step", values="adv1").round(3))

## 1. Probe: is EI's terminal regret predictable from the context?

In [ ]:
print("Spearman of the cross-fitted prediction with EI's terminal log10 regret, per decision step (priv: privileged references)")
display(probe.pivot(index="score", columns="step", values="spearman").round(3))
print("R2:"); display(probe[~probe.score.str.startswith("priv")].pivot(index="score", columns="step", values="r2").round(3))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), dpi=90)
for sc, g in probe.groupby("score"):
    if sc.endswith("/mlp"): continue
    ax[0].errorbar(g.step, g.spearman, yerr=[g.spearman - g.lo, g.hi - g.spearman], marker="o", ms=3, capsize=2, ls="--" if sc.startswith("priv") else "-", label=sc)
ax[0].set_xlabel("decision step t"); ax[0].set_ylabel("Spearman with EI's terminal regret"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3); ax[0].set_title("ridge per step, and privileged references")
for (a, b, l), g in pdiff.groupby(["a", "b", "learner"]):
    ax[1].errorbar(g.step + {"ridge": -.15, "mlp": .15}[l], g["diff"], yerr=[g["diff"] - g.lo, g.hi - g["diff"]], marker="o", ms=3, capsize=2, ls="-" if l == "ridge" else ":", label=f"{a} - {b} ({l})")
ax[1].axhline(0, color="k", lw=.8); ax[1].set_xlabel("decision step t"); ax[1].set_title("paired difference in Spearman"); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
plt.show()
display(pdiff.assign(ci=[f"{d:+.3f} [{lo:+.3f}, {hi:+.3f}]" for d, lo, hi in zip(pdiff["diff"], pdiff.lo, pdiff.hi)]).pivot(index=["a", "b", "learner"], columns="step", values="ci"))

## 2. Strata: where does UCB pay? Advantage by tercile of the predicted / the other seed's / the same seed's EI outcome

In [ ]:
for sc, g in strata[strata.option.str.endswith("_rest")].groupby("score"):
    print(sc, "- advantage of 'UCB for the rest' over EI (seed 0); tercile 0 = EI predicted / known to end best")
    display(g.pivot(index="tercile", columns="step", values="adv0").round(3))
g = strata[(strata.option.str.endswith("_k3")) & (strata.score == "priv:other_seed")]
print("priv:other_seed - 'UCB for 3 steps'"); display(g.pivot(index="tercile", columns="step", values="adv0").round(3))

## 3. Gates: gain over EI and over the static schedule

In [ ]:
def show(seed):
    s = summ[summ.seed == seed]
    t = s.assign(ci=[f"{m:+.3f} [{lo:+.3f}, {hi:+.3f}]" for m, lo, hi in zip(s["mean"], s.lo, s.hi)])
    out = t.pivot(index="variant", columns="contrast", values="ci")
    out["deviates"] = s.groupby("variant").deviates.first().round(2)
    return out
print("log10 terminal regret, positive = better; held-out tasks, 95% bootstrap CI")
print("seed 0 (deployed):"); display(show(0))
print("seed 1 (the decision executed on the other candidate set; not meaningful for priv:other_seed, whose score is that run):"); display(show(1))
print("gates chosen per fold (step -1 = never deviate; c = -inf: the ungated static schedule):")
display(gates.pivot(index="variant", columns="fold", values=["step", "option", "c"]).round(2))

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.

In [ ]:
print("pre-registered rule on", PRIMARY, "seed 0:", policy_verdict(summ, PRIMARY, WORTH))
c = summ[(summ.variant == "gate/priv:other_seed") & (summ.seed == 0) & (summ.contrast == "vs_static")].iloc[0]
print(f"ceiling (luck-corrected privileged gate) over static: {c['mean']:+.3f} [{c.lo:+.3f}, {c.hi:+.3f}]; below 0.05: {c.hi < 0.05}")